In [8]:
import os
import requests
import pandas as pd
from modules.config import DATA_DIR, HEADERS
from modules.utils import sleep_ratelimit
import time
BASE_MB_URL = "https://musicbrainz.org/ws/2"
BOB_DYLAN_MBID = "72c536dc-7137-4477-a521-567eeb840fa8"
HEADERS = {
    "User-Agent": "DylanCoverBot/0.1 (contact: suryaghoshal5+musicbrainz@gmail.com)"
}
LIMIT = 25  # number of results per page (MusicBrainz limit is 100)

# FINAL CODE

## STEP 1: GET ALL 1800 WORKS

In [35]:
import requests
import time
import csv
from datetime import datetime

BASE_MB = "https://musicbrainz.org/ws/2"
DYLAN_MBID = "72c536dc-7137-4477-a521-567eeb840fa8"  # Bob Dylan MBID
HEADERS = {
    "User-Agent": "DylanCoverProject/0.1 (contact@example.com)"
}


def fetch_dylan_works(limit=2500, output_csv="dylan_works_sample.csv"):
    offset = 0
    seen = 0
    page_size = 100  # max per request
    collected = []

    while seen < limit:
        remaining = limit - seen
        fetch_size = min(page_size, remaining)

        url = f"{BASE_MB}/work?artist={DYLAN_MBID}&limit={fetch_size}&offset={offset}&fmt=json&inc=aliases+tags+annotation"
        response = requests.get(url, headers=HEADERS)

        if response.status_code != 200:
            print(f"❌ Error fetching data: {response.status_code} - {response.text}")
            break

        data = response.json()
        works = data.get("works", [])
        if not works:
            print("⚠️ No more works found.")
            break

        for work in works:
            title = work.get("title", "")
            work_id = work.get("id", "")
            work_type = work.get("type", "")
            release_year = ""

            # Try to estimate earliest release year from "first-release-date" under ISWC or relations
            for rel in work.get("relations", []):
                if rel.get("type") == "first released in" and "begin" in rel.get("begin", ""):
                    release_year = rel["begin"][:4]

            collected.append({
                "Artist": "Bob Dylan",
                "Song Title": title,
                "Work ID": work_id,
                "Type": work_type,
                "Release Year": release_year
            })

        seen += len(works)
        offset += fetch_size
        time.sleep(1.1)

    # Save to CSV
    keys = ["Artist", "Song Title", "Work ID", "Type", "Release Year"]
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(collected)

    print(f"\n✅ Saved {len(collected)} Dylan works to {output_csv}")


if __name__ == "__main__":
    fetch_dylan_works()

⚠️ No more works found.

✅ Saved 1832 Dylan works to dylan_works_sample.csv


## Step 2: Get Recording IDs for all Work ID

In [40]:
import pandas as pd
import requests
import time
import csv

# MusicBrainz API config
BASE_URL = "https://musicbrainz.org/ws/2/recording"
HEADERS = {
    "User-Agent": "DylanRecordingCollector/1.0 (contact@example.com)"
}

# Respect API rate limit: 1 request/sec
RATE_LIMIT_DELAY = 1.1

# File paths
INPUT_CSV = "dylan_works_sample.csv"
OUTPUT_CSV = "dylan_recordings_output.csv"

def fetch_recordings(limit=2000):
    # Read input file
    df = pd.read_csv(INPUT_CSV)
    work_ids = df['Work ID'].dropna().unique()

    # Output CSV setup
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as out_file:
        writer = csv.writer(out_file)
        writer.writerow([
            "work_id",
            "recording_id",
            "recording_title",
            "recording_name",  # renamed disambiguation
            "first_release_date",
            "length"
        ])

        # Process only first `limit` work_ids
        for i, work_id in enumerate(work_ids[:limit]):
            print(f"[{i+1}/{min(limit, len(work_ids))}] 🎵 Fetching for work_id: {work_id}")

            try:
                url = f"{BASE_URL}?work={work_id}&fmt=json&limit=100"
                response = requests.get(url, headers=HEADERS)
                time.sleep(RATE_LIMIT_DELAY)

                if response.status_code == 200:
                    data = response.json()
                    recordings = data.get("recordings", [])
                    
                    for rec in recordings:
                        writer.writerow([
                            work_id,
                            rec.get("id", ""),
                            rec.get("title", ""),
                            rec.get("disambiguation", "NA") if rec.get("disambiguation") else "NA",
                            rec.get("first-release-date", "")[:4],
                            rec.get("length", "")
                        ])
                else:
                    print(f"⚠️ Error for work_id {work_id}: {response.status_code}")
            except Exception as e:
                print(f"❌ Exception for work_id {work_id}: {e}")

    print(f"\n✅ Done! Saved results to '{OUTPUT_CSV}'")

# Run
if __name__ == "__main__":
    fetch_recordings(limit=2000) 

[1/1832] 🎵 Fetching for work_id: 0079cb8f-d789-4b0f-9c5f-722e92d544f3
[2/1832] 🎵 Fetching for work_id: 0135740e-69dc-41a0-86d9-6a57664809a5
[3/1832] 🎵 Fetching for work_id: 0ae585bd-2294-4f6d-9ddb-80b6e8479a20
[4/1832] 🎵 Fetching for work_id: 0c8b9ba6-66bf-476c-82e2-122088d1d28d
[5/1832] 🎵 Fetching for work_id: 0db2e64f-8a3e-4ff5-b313-d6c997548fdf
[6/1832] 🎵 Fetching for work_id: 0fe3411d-afd0-4069-aa5f-f25fd5a9d6f5
[7/1832] 🎵 Fetching for work_id: 10cc574f-c2f0-43fb-8976-55a207db6240
[8/1832] 🎵 Fetching for work_id: 131b985c-662e-4f46-a90e-8b69e2235146
[9/1832] 🎵 Fetching for work_id: 1321a426-6138-44b7-babc-6954f2eaeb05
[10/1832] 🎵 Fetching for work_id: 145a08d2-3730-4888-9e3c-8f58fc1d307c
[11/1832] 🎵 Fetching for work_id: 1c4b258b-c6e0-43de-80f1-7bd6db5b7161
[12/1832] 🎵 Fetching for work_id: 210b7924-8db5-4e43-bf21-03f61ed90b1b
[13/1832] 🎵 Fetching for work_id: 267dcb7f-eb89-410c-ab60-9be6d506c757
[14/1832] 🎵 Fetching for work_id: 2fa90370-8e76-4cee-b554-828f53269782
[15/1832] 🎵 Fet

[115/1832] 🎵 Fetching for work_id: 1cdd7449-3aaf-4fe8-a860-f313f3c04680
[116/1832] 🎵 Fetching for work_id: 1fe69eac-d79c-3d11-83a3-abc14235081c
[117/1832] 🎵 Fetching for work_id: 228c6898-a609-48de-963a-0e8f4f7215d0
[118/1832] 🎵 Fetching for work_id: 240ec758-eb15-4008-aadc-9fdeb9eea608
[119/1832] 🎵 Fetching for work_id: 24ccb27d-461b-4211-8dbc-5b0cf1c61d63
[120/1832] 🎵 Fetching for work_id: 25cfaa19-cc04-44eb-93b3-777f1bf86edb
[121/1832] 🎵 Fetching for work_id: 2a9d6e19-baf1-49fa-abd0-42d8863bcb75
[122/1832] 🎵 Fetching for work_id: 2cf1d069-5469-4bab-ad02-bc015e7545c7
[123/1832] 🎵 Fetching for work_id: 36b9cbc1-b7cb-49e9-a414-97b90056a420
[124/1832] 🎵 Fetching for work_id: 3cf74125-6d20-442d-a0ff-79b7b891e0c3
[125/1832] 🎵 Fetching for work_id: 403c0d21-f45b-4e01-8a2b-833e41980ecc
[126/1832] 🎵 Fetching for work_id: 404b8a26-4cd4-4a85-b601-3f1ca29db48a
[127/1832] 🎵 Fetching for work_id: 45415531-0b1f-449b-98bf-3037fea78d12
[128/1832] 🎵 Fetching for work_id: 477e310a-e707-4e32-8751-b1d8f

[228/1832] 🎵 Fetching for work_id: 46b016b0-f025-4067-a13b-be2ff6a2b857
[229/1832] 🎵 Fetching for work_id: 4b9ce310-1b65-40f0-aaaa-f7e20de6050b
[230/1832] 🎵 Fetching for work_id: 4ecc764f-8013-440b-a062-46612b58a179
[231/1832] 🎵 Fetching for work_id: 50965013-d5bf-4191-87aa-eb33354cefe8
[232/1832] 🎵 Fetching for work_id: 54d505ad-da7b-36e8-8d43-94b7018e44de
[233/1832] 🎵 Fetching for work_id: 584483d2-ca00-3023-9967-6e9e91f4a165
[234/1832] 🎵 Fetching for work_id: 59c21df2-f1a3-49af-98e8-30becab2a5ce
[235/1832] 🎵 Fetching for work_id: 5a2f58f9-a9af-4ecc-9ffc-7e93c0f1196f
[236/1832] 🎵 Fetching for work_id: 5cee6e58-87c3-4183-b7b0-e2fa908e9240
[237/1832] 🎵 Fetching for work_id: 5ed18d5c-9848-393a-a919-1cc02ef4bfe8
[238/1832] 🎵 Fetching for work_id: 603c923f-f9f5-40c8-a9b0-d9f7340d04eb
[239/1832] 🎵 Fetching for work_id: 60958fd5-ea01-3536-ad17-ad664949a551
[240/1832] 🎵 Fetching for work_id: 62fe4b00-1685-4598-b18f-5a8984c8b51d
[241/1832] 🎵 Fetching for work_id: 66143d67-feac-4c2a-a301-edbd6

[341/1832] 🎵 Fetching for work_id: 5a5c6d05-8673-4a0a-838a-7aa200ea6ab9
[342/1832] 🎵 Fetching for work_id: 5bacade7-17bb-4b57-bc8d-4cac1f728b7f
[343/1832] 🎵 Fetching for work_id: 61ea103d-2716-38a8-891c-159af1057a5b
[344/1832] 🎵 Fetching for work_id: 629c83ae-58ea-4244-9cb9-d6ee32f89403
[345/1832] 🎵 Fetching for work_id: 67d3b639-57dd-41d2-8e35-a5e0c4ebe5c7
[346/1832] 🎵 Fetching for work_id: 6d5743f4-935d-48b9-a776-4e4da7b9bbd6
[347/1832] 🎵 Fetching for work_id: 6fe93887-6e71-4ee3-8cd8-e59c295b9a11
[348/1832] 🎵 Fetching for work_id: 7040d88f-6135-3917-8f8b-a01249d74c09
[349/1832] 🎵 Fetching for work_id: 72a1ee30-886c-4817-9f5c-2d900ab26209
[350/1832] 🎵 Fetching for work_id: 741f5ba8-5fee-43fa-a3c4-3eec2e44cc9a
[351/1832] 🎵 Fetching for work_id: 7b0dbb7a-0c74-4435-b4a6-a46d905d30ee
[352/1832] 🎵 Fetching for work_id: 80e22f92-a489-43a9-b905-02124b72f5d5
[353/1832] 🎵 Fetching for work_id: 83211b25-ab51-4668-91ff-917ee72c044c
[354/1832] 🎵 Fetching for work_id: 84a579fc-472d-4108-977a-4f15c

[454/1832] 🎵 Fetching for work_id: 90c5c3dc-6f0c-3137-a711-f9bca4a9f9b0
[455/1832] 🎵 Fetching for work_id: 93927ca2-50f3-4eef-a51e-0b4359967597
[456/1832] 🎵 Fetching for work_id: 94e8a507-cce5-3768-8374-1963a3b0da94
[457/1832] 🎵 Fetching for work_id: 99cdc47c-1492-4796-8fe7-ddda482cd693
[458/1832] 🎵 Fetching for work_id: 9d04d9b1-d474-4259-8a94-707f682011d1
[459/1832] 🎵 Fetching for work_id: 9dab33f3-a44e-4101-9b4f-f241b911a184
[460/1832] 🎵 Fetching for work_id: 9e132dbb-5058-484d-93c5-0ea794c5f99d
[461/1832] 🎵 Fetching for work_id: 9f8feacb-74d1-3520-b649-ff54ae9fbe06
[462/1832] 🎵 Fetching for work_id: a019f467-dcd4-426b-9822-aaa13cd47657
[463/1832] 🎵 Fetching for work_id: a3691c79-04cd-479d-90b7-8f2f62237ae9
[464/1832] 🎵 Fetching for work_id: a3fec692-0560-4571-9b3c-ffa0a9d72b24
[465/1832] 🎵 Fetching for work_id: a64b08e1-0259-4dfb-8b67-690fc3cf91fb
[466/1832] 🎵 Fetching for work_id: a9e69d23-bdca-4e4e-a402-1f9d5a96b9c3
[467/1832] 🎵 Fetching for work_id: abaca7ba-5d94-4ee1-8d91-8b683

...

[1351/1832] 🎵 Fetching for work_id: 8179b74c-9c34-4f8f-a46e-b375f6dfb34e
[1352/1832] 🎵 Fetching for work_id: 81833ad9-aa36-47e4-8c23-01208265e4fe
[1353/1832] 🎵 Fetching for work_id: 84aba04b-cb00-495c-8c84-9e66235774f2
[1354/1832] 🎵 Fetching for work_id: 88bc04a1-7f5f-3607-8855-56cfa1cfe6dc
[1355/1832] 🎵 Fetching for work_id: 88c462a9-253b-4352-a36e-8a837377f944
[1356/1832] 🎵 Fetching for work_id: 8bab4248-8fd8-3426-8cd4-cb08ce680d73
[1357/1832] 🎵 Fetching for work_id: 8ce50bf4-cbae-3fdd-8454-eb9e78974c02
[1358/1832] 🎵 Fetching for work_id: 8cfbaf6a-4e32-3038-9875-22b6b8e7f96e
[1359/1832] 🎵 Fetching for work_id: 9074d624-e8c2-4916-b237-4182f78250e0
[1360/1832] 🎵 Fetching for work_id: 93c371b8-a9f8-3984-b63d-7dd5b3958c21
[1361/1832] 🎵 Fetching for work_id: 96e9e5c2-0f54-3b8e-bd40-ad81d3378c83
[1362/1832] 🎵 Fetching for work_id: 9fc5e844-442e-3ad2-b377-c10419825fbb
[1363/1832] 🎵 Fetching for work_id: a1ef5deb-e877-4f71-a866-ec1c93ee25a2
[1364/1832] 🎵 Fetching for work_id: a2cd2ee6-4009-3

[1462/1832] 🎵 Fetching for work_id: a47e01f7-2fa2-454c-b4c4-f8529c6c0266
[1463/1832] 🎵 Fetching for work_id: a898e0f5-f48e-3ad3-be04-ee973956da3f
[1464/1832] 🎵 Fetching for work_id: a9fcefb4-e8ab-4878-870a-7241e0371dc9
[1465/1832] 🎵 Fetching for work_id: aa786654-a7a4-4bd5-bbcd-efb07cad2ff0
[1466/1832] 🎵 Fetching for work_id: abb09cc1-30f5-4752-a324-2993779ccdf6
[1467/1832] 🎵 Fetching for work_id: af1f7d6a-7760-428b-ad34-86d0c3947488
[1468/1832] 🎵 Fetching for work_id: b18a6708-7bd9-3580-a524-830f1fda6b3a
[1469/1832] 🎵 Fetching for work_id: b2cbea4c-7895-4ea2-b9ac-2909ad9f55af
[1470/1832] 🎵 Fetching for work_id: b2f9b96d-89a1-4499-9606-26e60860309d
[1471/1832] 🎵 Fetching for work_id: b391eba1-24bc-3d4b-9510-e36f4ad0e4f5
[1472/1832] 🎵 Fetching for work_id: b472938b-ab20-4464-9cb2-61c7b0c22d86
[1473/1832] 🎵 Fetching for work_id: b5da1d13-ba77-47fc-b8b7-484283d9c214
[1474/1832] 🎵 Fetching for work_id: b7722506-f8e3-4964-9930-acec95439779
[1475/1832] 🎵 Fetching for work_id: b99fd41d-d051-3

[1573/1832] 🎵 Fetching for work_id: aa15f9e2-7766-43d8-b9a1-4db1865f84cd
[1574/1832] 🎵 Fetching for work_id: ac967d83-36c2-3bf5-a2a6-562c4b1fdb2e
[1575/1832] 🎵 Fetching for work_id: b1d08151-06bc-410c-acbd-00acda64e80b
[1576/1832] 🎵 Fetching for work_id: b70afde7-4f1a-3fe6-868d-14947c686192
[1577/1832] 🎵 Fetching for work_id: b7711877-bcc9-4716-979d-2c07c142cdf2
[1578/1832] 🎵 Fetching for work_id: bb010dc2-bb68-4a18-b519-bf83094bb24d
[1579/1832] 🎵 Fetching for work_id: bdaf022d-5678-4a95-9aa6-45eda0376f56
[1580/1832] 🎵 Fetching for work_id: beb11edc-7217-3245-950c-f1e6426eee4f
[1581/1832] 🎵 Fetching for work_id: c05e06fd-89d8-36d8-8aab-9804b8dd5d89
[1582/1832] 🎵 Fetching for work_id: ca8235c6-1891-4d31-b55e-49f51ba3e613
[1583/1832] 🎵 Fetching for work_id: cc132527-02e4-4210-994e-6431784a25e4
[1584/1832] 🎵 Fetching for work_id: d3109c6d-7631-4b93-9f93-298077b7df9c
[1585/1832] 🎵 Fetching for work_id: d45cd2ac-52d2-3469-b9c3-985dc15523ef
[1586/1832] 🎵 Fetching for work_id: d6e19860-7980-4

[1684/1832] 🎵 Fetching for work_id: e3c66cea-bf7b-3748-a95e-099ab20b5c71
[1685/1832] 🎵 Fetching for work_id: e3c9e4e1-523a-4c90-949c-cec79d39872e
[1686/1832] 🎵 Fetching for work_id: e409f0f4-26cf-43dc-a6ab-6c625bb9dfc4
[1687/1832] 🎵 Fetching for work_id: e581da55-cc7a-4d79-8dc7-90adbade453f
[1688/1832] 🎵 Fetching for work_id: e9e70db4-4510-4795-9baa-93adbd9dbba9
[1689/1832] 🎵 Fetching for work_id: ecb183f8-dd48-4f56-80fe-4e7684363e12
[1690/1832] 🎵 Fetching for work_id: ecf63919-978c-4705-b6f1-d31a5d9d21a1
[1691/1832] 🎵 Fetching for work_id: ee6702ce-cdca-47b9-87de-c9e9b7d24a10
[1692/1832] 🎵 Fetching for work_id: eef1dd01-7be0-3402-97e5-16fdd198ef2a
[1693/1832] 🎵 Fetching for work_id: ef19d4c4-7e50-4846-b964-202d2ce51097
[1694/1832] 🎵 Fetching for work_id: f37d593d-c4e9-47b3-9990-b827be5cd485
[1695/1832] 🎵 Fetching for work_id: f3be635d-d7b0-465d-a8df-50656932c7f9
[1696/1832] 🎵 Fetching for work_id: f7a91d35-6e96-4169-8d36-1796e4a94401
[1697/1832] 🎵 Fetching for work_id: f9026f2d-a8aa-3

[1795/1832] 🎵 Fetching for work_id: f617eeb9-b07f-4174-a75f-80bf46cec9ed
[1796/1832] 🎵 Fetching for work_id: f6a7551b-875d-4db9-ad68-7e6b79c9d7e4
[1797/1832] 🎵 Fetching for work_id: f96c8eb5-2d62-4dbc-a3ed-a2aa8caf3557
[1798/1832] 🎵 Fetching for work_id: fcaf30bd-dd45-40d3-828e-b30922a248b1
[1799/1832] 🎵 Fetching for work_id: ff6f7104-a5c6-49e2-bc19-77c14dc5502e
[1800/1832] 🎵 Fetching for work_id: fff347bb-d059-4cb0-94ca-86e83e533a9a
[1801/1832] 🎵 Fetching for work_id: 144876ae-136e-4fd4-844b-2bc77e3bd476
[1802/1832] 🎵 Fetching for work_id: 1ed2d03a-79bd-4f10-b7f3-5990919c35e6
[1803/1832] 🎵 Fetching for work_id: 20ca3866-3a67-4fa2-bc1c-64a3ace19782
[1804/1832] 🎵 Fetching for work_id: 25af0e8f-3040-49e5-83c6-26e390c901ad
[1805/1832] 🎵 Fetching for work_id: 2d3373a9-3b74-350d-a10d-877812552aae
[1806/1832] 🎵 Fetching for work_id: 41d32060-dba1-4010-94a2-b31b7448cef2
[1807/1832] 🎵 Fetching for work_id: 50214f6e-87fd-4067-9a5f-b3e8e1b748c5
[1808/1832] 🎵 Fetching for work_id: 54dc9f1f-046d-3

## Step 3: Get Release ID and Artist Details for all recording ids

In [41]:
import pandas as pd
import requests
import time
import csv

BASE_URL = "https://musicbrainz.org/ws/2/recording/"
HEADERS = {
    "User-Agent": "DylanCoverProject/2.0 (contact@example.com)"
}
RATE_LIMIT_DELAY = 1.1

INPUT_FILE = "dylan_recordings_output.csv"
OUTPUT_FILE = "dylan_recording_metadata.csv"

def enrich_recordings(limit=100):
    # Read input
    df = pd.read_csv(INPUT_FILE)
    recording_ids = df['recording_id'].dropna().unique()

    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as out_file:
        writer = csv.writer(out_file)
        header = [
            "recording_id", "recording_title", "recording_name", "first_release_date", "length",
            "artist_name_1", "artist_id_1",
            "artist_name_2", "artist_id_2",
            "artist_name_3", "artist_id_3",
            "artist_name_4", "artist_id_4",
            "artist_name_5", "artist_id_5",
            "release_id", "release_title", "release_count",
            "external_urls"
        ]
        writer.writerow(header)

        for idx, rec_id in enumerate(recording_ids[:limit]):
            print(f"[{idx+1}/{limit}] Fetching recording ID: {rec_id}")
            try:
                url = f"{BASE_URL}{rec_id}?fmt=json&inc=artist-credits+releases+url-rels"
                response = requests.get(url, headers=HEADERS)
                time.sleep(RATE_LIMIT_DELAY)

                if response.status_code != 200:
                    print(f"⚠️ Error: {response.status_code} for ID {rec_id}")
                    continue

                data = response.json()

                # Basic recording fields
                title = data.get("title", "")
                disambig = data.get("disambiguation", "NA") or "NA"
                release_date = data.get("first-release-date", "")
                length = data.get("length", "")

                # Artist credits
                artist_credits = data.get("artist-credit", [])
                artists_flat = []
                for ac in artist_credits[:5]:
                    artist = ac.get("artist", {})
                    artists_flat.extend([
                        artist.get("name", "NA"),
                        artist.get("id", "NA")
                    ])
                while len(artists_flat) < 10:
                    artists_flat.extend(["NA", "NA"])

                # Release info
                releases = data.get("releases", [])
                first_release = releases[0] if releases else {}
                release_id = first_release.get("id", "NA")
                release_title = first_release.get("title", "NA")
                release_count = len(releases)

                # URL relations
                urls = []
                relations = data.get("relations", [])
                for rel in relations:
                    url_obj = rel.get("url", {})
                    resource = url_obj.get("resource")
                    if resource:
                        urls.append(resource)
                external_urls = "|".join(urls) if urls else "NA"

                # Final row
                row = [
                    rec_id, title, disambig, release_date, length,
                    *artists_flat,
                    release_id, release_title, release_count,
                    external_urls
                ]
                writer.writerow(row)

            except Exception as e:
                print(f"❌ Exception for recording ID {rec_id}: {e}")

    print(f"\n✅ Done. Output saved to: {OUTPUT_FILE}")

# Run for 100 recordings
if __name__ == "__main__":
    enrich_recordings(limit=10)

[1/10] Fetching recording ID: 31815d0f-ef60-4717-8381-55867d53f7fb
[2/10] Fetching recording ID: 389a2789-dbee-4256-843d-510e9dc03c48
[3/10] Fetching recording ID: 46d3c6d3-c109-405d-a5e7-670654ed0a1b
[4/10] Fetching recording ID: 729d6ce6-a75b-48f7-b6e0-2f559888bc24
[5/10] Fetching recording ID: 860874ff-df30-47dd-b889-10d8a80513ee
[6/10] Fetching recording ID: a6869dda-4f45-40fa-8cb6-6392ffd34824
[7/10] Fetching recording ID: eef76715-4643-4140-ac40-d728c403c18f
[8/10] Fetching recording ID: 0446e8b6-24f2-4240-9cf9-499c4b0f932d
[9/10] Fetching recording ID: 07466cc9-e504-46cd-b3af-20c4385e0de2
[10/10] Fetching recording ID: 09a6f18b-3768-4a69-9237-6e6d8aae4db8

✅ Done. Output saved to: dylan_recording_metadata.csv


In [ ]:

https://musicbrainz.org/ws/2/work/97df5c91-6a29-4b1d-8c3a-f26b1a0406e8?fmt=json&inc=artist-rels+url-rels+recording-rels